In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb

from scipy import stats
from sklearn.base import clone
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1. Dataset loading and splitting

In [ ]:
data = pd.read_csv("electricity_practice.csv")
data["time"] = pd.to_datetime(data["time"], utc=True)
data = data.set_index("time").sort_index().asfreq("h")

train, test = train_test_split(data, test_size=0.2, shuffle=False)

print("Dataset shape: ", data.shape)
print(f"Development period: {train.index[0]} to {train.index[-1]}")
print(f"Holdout period: {test.index[0]} to {test.index[-1]}")
display(train.head())

# 2. Inspect data quality

In [ ]:
quality = train.describe().T
quality["missing_pct"] = train.isna().mean() * 100
quality["unique_values"] = train.nunique()
quality["negative_values"] = (train < 0).sum(axis=0)

display(quality)

## 2.1 Clean anomalies and invalid values

In [ ]:
train.plot.line(
    subplots=True,
    sharex=False,
    layout=(5,4),
    figsize=(14,10),
    legend=False,
    title=train.columns.to_list()
)
plt.tight_layout()
plt.show()

In [ ]:
# Apply fixed cutoff to clean out obvious extreme high values
cutoffs = {
    "Thermal_Methane": 4500,
    "Waste": 1000,
    "Thermal_Lignite": 480,
    "Solar": 20000,
    "Wind_Onshore": 25000,
    "Wind_Offshore": 10000,
    "Nuclear": 10000,
    "Hydro_Run_of_River": 2500,
    "Thermal_Biomass": 6000,
    "Thermal_Hard_Coal": 7000,
    "Geothermal": 50,
}

# Sources with only zeros 
zero_cols = ["Thermal_Coal_Derived_Gas", 'Thermal_Oil_Shale', 'Thermal_Peat', 'Marine']
# Sources with nearly constant values
const_cols = ["Other_Generation", "Other_Renewables"]

def clean_anomalies(df):
    cleaned = df.copy()

    # Mask obvious anomalies
    for column, cutoff in cutoffs.items():
        cleaned[column] = cleaned[column].mask(cleaned[column] > cutoff)

    # Mask invalid negative generation 
    cleaned = cleaned.mask(cleaned < 0)

    return cleaned.drop(columns = zero_cols + const_cols)

train_clean = clean_anomalies(train)
test_clean = clean_anomalies(test)

In [ ]:
# Detect suspicious spikes across columns
prev_ = train_clean.shift(1)
next_ = train_clean.shift(-1)

neighbor_mean = (prev_ + next_) / 2
typical_change = train_clean.diff().abs().median()

spikes_up = (
    (neighbor_mean > 0) & 
    (train_clean > 3 * neighbor_mean) & 
    (train_clean - prev_ > 10 * typical_change) & 
    (train_clean - next_ > 10 * typical_change) 
)

# spikes_down = (
#     (neighbor_mean > 0) & 
#     (train_clean > 0) & 
#     (train_clean < neighbor_mean / 3) & 
#     (prev_ - train_clean > 10 * typical_change) & 
#     (next_ - train_clean > 10 * typical_change) 
# )

spikes = spikes_up # | spikes_down

# Number of spikes per column
spike_summary = pd.DataFrame({
    "Up": spikes_up.sum(),
    # "Down": spikes_down.sum(),
})

spike_summary['Total'] = spike_summary['Up'] #+ spike_summary['Down']
display(spike_summary.sort_values('Total', ascending=False))

# Inspect specific spikes with around time points
spike_details = train_clean.where(spikes).stack().reset_index()
spike_details.columns = ["Time", "Source", "Value"]

review = spike_details[spike_details["Source"] != "Thermal_Oil"]

for _, row in review.iterrows():
    time, col = row["Time"], row["Source"]
    display(
        train_clean.loc[
            time - pd.Timedelta(hours=2): 
            time + pd.Timedelta(hours=2),
            col
        ]
    )

train_clean.loc[['2015-04-26 19:00', '2015-07-01 06:00'], "Hydro_Reservoir"] = np.nan
train_clean.loc['2018-01-29 14:00', 'Thermal_Lignite'] = np.nan
train_clean.loc['2015-11-26 20:00', 'Thermal_Hard_Coal'] = np.nan
train_clean.loc['2017-05-10 15:00', 'Wind_Offshore'] = np.nan

# Show missing pct of different sources after cleaning
display((train_clean.isna().mean() * 100).round(2).rename("missing_pct"))

## 2.2 Check missing values

In [ ]:
# Check missing patterns after cleaning
missing_summary = []

for col in train_clean.columns:
    missing = train_clean[col].isna()

    # Identify consecutive missing gaps
    groups = (missing != missing.shift()).cumsum()
    gaps = missing.groupby(groups).sum()
    gaps = gaps[gaps > 0]

    # Count short gaps
    short_count = int(gaps.le(3).sum())
    short_pct = 100 * short_count / len(gaps) if len(gaps) else 0

    # Find the longest missing gap and its dates
    if len(gaps) > 0:
        longest = int(gaps.max())
        dates = train_clean.index[groups == gaps.idxmax()]
        start, end = dates[0], dates[-1]
    else:
        longest = 0
        start, end = pd.NaT, pd.NaT

    missing_summary.append({
        "Column": col,
        "Original Missing %": train[col].isna().mean() * 100,
        "After Cleaning %": missing.mean() * 100,
        "Missing Count": missing.sum(),
        "Number of Gaps": len(gaps),
        "Short Gaps (1-3 hours)": short_count,
        "Short Gap %": short_pct,
        "Longest Gap (hours)": longest,
        "Gap Start": start,
        "Gap End": end
    })

# Summary table
missing_summary = pd.DataFrame(missing_summary).set_index("Column")
missing_summary = missing_summary.sort_values("After Cleaning %", ascending=False)

display(missing_summary.round(2))

# Overall missingness
print("Hours with any missing:", train_clean.isna().any(axis=1).sum())
print("Hours with all sources missing:", train_clean.isna().all(axis=1).sum())

# # Check whether any hourly timestamps are absent
# expected_hours = pd.date_range(
#     train_clean.index.min(), train_clean.index.max(), freq="h"
# )
# print("Missing hourly timestamps:", len(expected_hours.difference(train_clean.index)))

In [ ]:
# Check autocorr of different sources for forward filling
for col in train_clean:
    cutoff = 0

    for lag in range(1, 25):
        if train_clean[col].autocorr(lag) >= 0.95:
            cutoff = lag
        else:
            break

    print(f"{col}: {cutoff} hours")

# Part 1

## Q1: Source trend, seasonality, data issue

Use cleaned development data only. Daily averages reveal the long-run movements, while monthly and hourly averages show recurring seasonality.

### a: Trend analysis

In [ ]:
daily_sources = train_clean.resample("D").mean()
daily_counts = train_clean.resample("D").count()

daily_sources = daily_sources.where(daily_counts >= 12)

trend = daily_sources.rolling(180, min_periods=90).mean()
trend.plot.line(
    subplots=True,
    sharex=False,
    layout=(4,4),
    figsize=(14,10),
    legend=False,
    title=train_clean.columns.to_list()
)
plt.suptitle("Trend analysis of generation by source")
plt.tight_layout()
plt.show()


### b: Annual and intraday seasonality

In [ ]:
daily_sources.plot.line(
    subplots=True,
    sharex=False,
    layout=(4,4),
    figsize=(14,10),
    legend=False,
    title=train_clean.columns.to_list()
)
plt.suptitle("Daily seasonality of generation by source")
plt.tight_layout()
plt.show()

In [ ]:
# Monthly seasonality
monthly_sources = train_clean.groupby(train_clean.index.month).mean()

monthly_sources.plot.line(
    subplots=True,
    sharex=False,
    layout=(4,4),
    figsize=(14,10),
    legend=False,
    title=train_clean.columns.to_list()
)
plt.suptitle("Monthly seasonality of generation by source")
plt.tight_layout()
plt.show()

In [ ]:
# Intraday seasonality
hourly_sources = train_clean.groupby(train_clean.index.hour).mean()

hourly_sources.plot.line(
    subplots=True,
    sharex=False,
    layout=(4,4),
    figsize=(14,10),
    legend=False,
    title=train_clean.columns.to_list()
)
plt.suptitle("Hourly seasonality of generation by source")
plt.tight_layout()
plt.show()

### Q1 finding summary

**1. Long-Term Trends**

Different generation sources exhibit distinct long-term trends.

- **Wind**, **Solar**, **Thermal**, **Nuclear** 
  
**2. Annual Seasonality**

Several generation sources exhibit clear seasonal patterns.

- **Solar**, **Hydro**, **Thermal Biomass**, **Wind** 
  
**3. Intraday Patterns**

Generation also varies by hour, although the strength of the pattern differs across sources.

- **Solar**, **Pumped storage**
- Other sources more stable hourly generation

**4. Data Quality Issues**

Several potential data-quality issues were identified:

- **Missing and negative observations**, **Extreme values**, **Isolated spikes**, **Structural changes**, **Incomplete days** 

**Conclusion**

Patterns vary substantially across sources. Renewable sources such as Solar, Wind, and Hydro show strong weather-related variability and seasonality. Conventional generation sources exhibit different levels of operational stability and longer-term structural changes.

The dataset also contains missing observations, extreme values, and possible reporting anomalies. Careful cleaning is important, but unusually high or low generation should not automatically be treated as erroneous because it may reflect genuine operational behavior.

## Q2: Daily solar trend, seasonality, distribution

In [ ]:
# Analyze trend and seasonality
solar = train_clean["Solar"]
solar_hours = solar.resample("D").count()
solar_daily = solar.resample("D").mean().where(solar_hours >= 12).dropna()

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
solar_daily.plot(ax=axes[0, 0], title="Daily average Solar")
solar_daily.rolling(180, min_periods=90).mean().plot(
    ax=axes[0, 1], title="180-day rolling trend")
solar_daily.groupby(solar_daily.index.month).mean().plot(
    ax=axes[1, 0], title="Average Solar by month")
solar.groupby(solar.index.hour).mean().plot(
    ax=axes[1, 1], title="Average Solar by UTC hour")
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of daily solar generation
solar_clean = solar_daily.dropna()

# 1. Deseasonalize solar by month
solar_monthly = solar_clean.groupby(solar_clean.index.month).transform("mean")
solar_deseason = solar_clean - solar_monthly + solar_clean.mean()

# 2. Apply yeo-johnson transformation to both series
solar_daily_pt = PowerTransformer(method="yeo-johnson").fit_transform(solar_clean.to_frame())
solar_deseason_pt = PowerTransformer(method="yeo-johnson").fit_transform(solar_deseason.to_frame())

solar_daily_pt = pd.Series(solar_daily_pt.flatten(), index=solar_clean.index)
solar_deseason_pt = pd.Series(solar_deseason_pt.flatten(), index=solar_clean.index)

# 3. Compare distribution
solar_comparisons = {
    "Original": solar_clean,
    "Yeo-Johnson (Original)": solar_daily_pt,
    "Deseasonalized": solar_deseason, 
    "Yeo-Johnson (Deseasonalized)": solar_deseason_pt
} 

fig, axes = plt.subplots(4, 2, figsize=(12,12))

for i, (name, values) in enumerate(solar_comparisons.items()):
    values.plot.hist(bins=30, ax=axes[i, 0], title=name)
    stats.probplot(values, dist="norm", plot=axes[i,1])
    axes[i,1].set_title(f"Q-Q plot of {name}")
plt.tight_layout()
plt.show()

# 4. Compare skewness and kurtosis
results = [["Normal", 0, 3]]

for name, values in solar_comparisons.items():
    results.append([name, values.skew(), values.kurt()])

solar_summary = pd.DataFrame(
    results, columns=["Transformation", "Skewness", "Kurtosis"]
)
display(solar_summary.round(2))

### Q2 finding summary

**1. Trend and Seasonality**

Strong annual seasonality, with recurring peaks in summer and troughs in winter. Longer daylight hours and stronger sunlight during summer increase solar. A gradual upward trend over time, potentially reflecting increased solar generation capacity.

**2. Distribution and Normality**

The original not normally distributed and exhibits positive skewness. Partly due to seasonality: many winter days have relatively low generation, while summer days can have substantially higher generation. Weather variability also contributes.

**3. Deseasonalization**

Removing monthly averages reduces the differences between s&w generation, distribution more symmetric. Deseasonalization does not necessarily produce a normal distribution because short-term weather fluctuations, extreme observations, and long-term trends remain. 

**Conclusion**

Solar has clear annual seasonality and a non-normal distribution. Deseasonalization addresses systematic monthly differences but cannot guarantee normality, as remaining deviations in the Q-Q plots may indicate.

## Q3: Hourly pumped storage properties

In [ ]:
pumped = train_clean["Pumped_Storage"]
hourly_pumped = pumped.groupby(pumped.index.hour).agg(
    ["count", "mean", "median", "std", "min", "max"]
)
hourly_pumped["zero_pct"] = pumped.groupby(pumped.index.hour).apply(
    lambda x: 100 * (x.dropna() == 0).mean()
)
display(hourly_pumped.round(2))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7))

hourly_pumped["mean"].plot(ax=axes[0], title="Mean pumped-storage generation by hour", marker="o")
axes[0].set(xlabel="Hour", ylabel="Average generation")

hourly_pumped["zero_pct"].plot.bar(ax=axes[1], title="Zero-generation rate by hour")
axes[1].set(xlabel="Hour", ylabel="Zero observations (%)")

plt.tight_layout()
plt.show()

print("Peak before noon:", hourly_pumped.loc[:11, "mean"].idxmax())
print("Peak from noon onward:", hourly_pumped.loc[12:, "mean"].idxmax())

### Q3 finding summary

**1. Intraday Pattern**

Pumped storage shows two clear generation peaks, around 9 am and 7 pm (high-demand hours), with the evening peak generally higher. Generation is relatively low overnight and around midday. 

**2. Zero Generation and Variability**

Frequent zero-generation periods and high variability => it operates intermittently not providing constant electricity generation.

**3. Economic Interpretation**

Consumes electricity to pump water when prices are low and generates electricity when prices are high.

- Morning: Higher demand may encourage pumped-storage generation.

- Midday: Strong Solar generation can increase electricity supply and lower prices, reducing the need for pumped-storage generation and potentially creating opportunities for pumping.

- Evening: Solar generation declines while demand remains relatively high, increasing the need for flexible generation from pumped storage.

**Conclusion**

Pumped storage exhibits a clear intraday pattern, consistent with its role as a flexible energy-storage resource that helps balance electricity supply and demand.

# Part 2

## 1. Construct target

In [ ]:
thermal_cols = [col for col in train_clean.columns if col.startswith("Thermal_")]
y_dev = train_clean[thermal_cols].sum(axis = 1, min_count = len(thermal_cols))
y_holdout = test_clean[thermal_cols].sum(axis = 1, min_count = len(thermal_cols))

print("Target components:", thermal_cols)
print("Valid development targets:", y_dev.notna().sum(), "in", len(y_dev))
print("Valid test targets:", y_holdout.notna().sum(), "in", len(y_holdout))

fig, axes = plt.subplots(1, 2, figsize = (12, 4))
y_dev.resample("D").mean().plot(ax = axes[0], title = "Daily average total Thermal")
y_dev.groupby(y_dev.index.hour).mean().plot(ax = axes[1], title="Average total Thermal by hour")
plt.tight_layout()
plt.show()

In [ ]:
# # 1. Prepare data (exclude Thermal components to avoid target leakage)
# corr_data = train_clean.drop(columns = thermal_cols).copy()
# corr_data["Thermal"] = y_dev

# # 2. Hourly correlation with Thermal
# hourly_corr = corr_data.corr(method="spearman")["Thermal"].drop("Thermal")

# # 3. Daily correlation with Thermal
# daily_data = corr_data.resample("D").mean()

# # Exclude daily means based on fewer than 12 valid hours
# daily_counts = corr_data.resample("D").count()
# daily_data = daily_data.where(daily_counts >= 12)

# daily_corr = daily_data.corr(method="spearman")["Thermal"].drop("Thermal")

# # 4. Compare hourly and daily correlations
# corr_summary = pd.DataFrame({
#     "Hourly": hourly_corr,
#     "Daily": daily_corr
# })

# corr_summary = corr_summary.sort_values("Hourly", key = abs, ascending = False)

# display(corr_summary.round(3))

# # 5. Check correlations between potential predictors
# display(corr_data.corr().round(2))


## 2. Construct features

In [ ]:
sources = pd.concat([train_clean, test_clean]).sort_index()
features = sources[["Solar", "Wind_Onshore", "Wind_Offshore"]].copy()

for col in ["Pumped_Storage", "Hydro_Run_of_River"]:
    features[f"{col}_lag1"] = sources[col].shift(1)  # previous hour only

hours = features.index.hour
for k in [1, 2]:
    features[f"hour_sin{k}"] = np.sin(2 * np.pi * k * hours / 24) # 24-hour cycle
    features[f"hour_cos{k}"] = np.cos(2 * np.pi * k * hours / 24) # 12-hour cycle

year_days = np.where(features.index.is_leap_year, 366, 365)
annual_phase = (features.index.dayofyear - 1) / year_days
features["year_sin"] = np.sin(2 * np.pi * annual_phase)
features["year_cos"] = np.cos(2 * np.pi * annual_phase)

# For imputation only, will be dropped later
features["month"] = features.index.month  
features["hour"] = features.index.hour    

# features["trend"] = (
#     (features.index - features.index[0]).total_seconds()
#     / (365.25 * 24 * 3600)
# )

X_dev = features.loc[train_clean.index].copy()
X_holdout = features.loc[test_clean.index].copy()
print("Feature columns:", features.columns.to_list())

## 3. Impute features

In [ ]:
def impute_pair(train_features, other_features):
    X_train = train_features.copy()
    X_other = other_features.copy()

    # If need to ffill some columns
    # col = "Pumped_Storage_lag1"
    # X_train[col] = X_train[col].ffill(limit=3)
    # X_other[col] = X_other[col].ffill(limit=3)

    rules = {
        "Solar": ["month", "hour"],
        "Wind_Onshore": ["month", "hour"],
        "Wind_Offshore": ["month"],
        "Hydro_Run_of_River_lag1": ["month"],
        "Pumped_Storage_lag1": ["hour"],
    }

    for col, group_keys in rules.items():
        group_mean = train_features.groupby(group_keys)[col].mean().rename("group_mean")
        overall_mean = train_features[col].mean()
        for frame in (X_train, X_other):
            by_group = frame.join(group_mean, on = group_keys)["group_mean"]
            frame[col] = frame[col].fillna(by_group).fillna(overall_mean)

    return (X_train.drop(columns = ["month", "hour"]),
            X_other.drop(columns = ["month", "hour"]))

## 4. Model comparison with expanding-window cross validation

In [ ]:
models = {
    "OLS": LinearRegression(),

    "Ridge": Ridge(alpha=10),
    
    "Random forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=3,
        min_samples_leaf=50,
        random_state=42,
    ),

    "XGBoost": xgb.XGBRegressor(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.03,
        random_state=42,
    )
}

cv = TimeSeriesSplit(n_splits = 3, test_size = 24 * 365)
cv_rows = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_dev), start = 1):

    X_train, X_val = impute_pair(X_dev.iloc[train_idx], X_dev.iloc[val_idx])
    y_train, y_val = y_dev.iloc[train_idx], y_dev.iloc[val_idx]
    train_valid, val_valid = y_train.notna(), y_val.notna()

    for name, model in models.items():
        if name == "Ridge" or name == "OLS":
            scaler = StandardScaler().set_output(transform="pandas")
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

        model.fit(X_train.loc[train_valid], y_train.loc[train_valid])

        train_pred = model.predict(X_train.loc[train_valid])
        val_pred = model.predict(X_val.loc[val_valid])

        cv_rows.append({
            "Fold": fold, "Model": name,
            "RMSE": root_mean_squared_error(y_val.loc[val_valid], val_pred),
            "MAE": mean_absolute_error(y_val.loc[val_valid], val_pred),
            "R2": r2_score(y_val.loc[val_valid], val_pred),
        })

    print(f"Fold {fold}: validation {X_val.index.min().date()} to {X_val.index.max().date()}")

### Model evaluation

In [ ]:
cv_results = pd.DataFrame(cv_rows)
display(cv_results.set_index(["Fold", "Model"]).round(2))

cv_summary = (cv_results.groupby("Model")[["RMSE", "MAE", "R2"]].mean().sort_values("RMSE"))
display(cv_summary.round(2))

best_name = cv_summary.index[0]
print("Selected model:", best_name)

## 5. Refit on development and feature importance

In [ ]:
X_fit, X_test = impute_pair(X_dev, X_holdout)
fit_valid = y_dev.notna()
final_model = clone(models[best_name])
final_model.fit(X_fit.loc[fit_valid], y_dev.loc[fit_valid])

# If best model is OLS/ridge
# values = final_model.coef_ * X_fit.std(ddof=0)
# importance_label = "Thermal change per 1 SD increase in feature"

values = final_model.feature_importances_
importance_label = "Relative feature importance"

importance = pd.Series(values, index = X_fit.columns)
display(importance.reindex(importance.abs().sort_values(ascending = False).index).round(3))
importance.loc[importance.abs().nlargest(10).index].sort_values().plot.barh(
    figsize = (9, 5), title = "top feature effects")
plt.xlabel(importance_label)
plt.tight_layout()
plt.show()

## 6. Heldout prediction

In [ ]:
test_predictions = pd.Series(
    final_model.predict(X_test), index = X_holdout.index
)
valid_test = y_holdout.notna()
actual = y_holdout.loc[valid_test]
predicted = test_predictions.loc[valid_test]
baseline = np.repeat(y_dev.dropna().mean(), len(actual))

def evaluate(true, pred):
    return {
        "RMSE": root_mean_squared_error(true, pred),
        "MAE": mean_absolute_error(true, pred),
        "R2": r2_score(true, pred),
    }

final_scores = pd.DataFrame({
    "Train Mean Baseline": evaluate(actual, baseline), 
    best_name: evaluate(actual, predicted),
}).T

display(final_scores.round(3))
print("Scored hours:", valid_test.sum(), 'in', len(valid_test))
print("Excluded hours with missing/invalid Thermal target:", (~valid_test).sum())

In [ ]:
comparison = pd.DataFrame({"Actual": actual, "Predicted": predicted})
comparison.plot(
    figsize = (12, 4), title = "Holdout daily average Thermal generation"
)
plt.ylabel("Average generation")
plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.DataFrame({"Actual": y_holdout, "Predicted": predicted})
comparison.resample("D").mean().plot(
    figsize = (12, 4), title = "Holdout daily average Thermal generation"
)
plt.ylabel("Average generation")
plt.tight_layout()
plt.show()